# 🚀 LLM Post-Training Masterclass: From Base Model to Reasoning Assistant
### Complete Hands-On Guide with `Qwen/Qwen2.5-0.5B` Base Model
**Author & Stack**: Antigravity AI Engineering Lab | PyTorch 2.x • HuggingFace • MPS / CUDA

---

## 📌 Why This Notebook Exists
In modern AI, **95% of compute is spent on pre-training, but 100% of human interaction is governed by post-training.**

A raw **Base Model** is not an assistant:
- Prompt: `"What is the capital of France?"`
- Base Model: `"
What is the capital of Germany?
What is the capital of Italy?
List of European capitals..."`

A base model is an **unsupervised token continuation engine**. It does not know how to converse, refuse harmful requests, follow system guidelines, or reason in chain-of-thought steps.

**Post-training is the discipline of turning this raw autocomplete simulator into a polite, truthful, and reasoning intelligent agent.**

```
                                  THE POST-TRAINING PIPELINE
                                  
   ┌──────────────────────┐
   │  Pre-trained Base    │   (e.g., Qwen2.5-0.5B, trained on 18T tokens)
   │  Language Model      │   Raw next-token prediction, autocomplete chaos
   └──────────┬───────────┘
              │
              ▼   [Stage 1: Special Tokens & ChatML Formatting]
   ┌──────────────────────┐
   │  Chat Templates      │   Inject <|im_start|>system/user/assistant<|im_end|>
   └──────────┬───────────┘
              │
              ▼   [Stage 2: Supervised Fine-Tuning (SFT) + Loss Masking]
   ┌──────────────────────┐
   │  SFT + LoRA Model    │   Teacher-forcing cross-entropy ONLY on assistant tokens
   │  (Instruction Model) │   Learns the conversation interface & QA style
   └──────────┬───────────┘
              │
              ├─────────────────────────────────────────┐
              ▼   [Stage 3: Preference Alignment]       ▼   [Stage 4: Reasoning RL]
   ┌──────────────────────┐                  ┌──────────────────────┐
   │  DPO (Direct         │                  │  GRPO (Group Relative│
   │  Preference Opt.)    │                  │  Policy Optimization)│
   │  Bradley-Terry loss  │                  │  Verifiable reward   │
   │  Chosen vs Rejected  │                  │  DeepSeek-R1 style   │
   └──────────┬───────────┘                  └──────────┬───────────┘
              │                                         │
              └────────────────────┬────────────────────┘
                                   │
                                   ▼
                      ┌─────────────────────────┐
                      │ Production-Ready Agent  │
                      │ Aligned & Verifiable    │
                      └─────────────────────────┘
```

---

## 🧭 Master Table of Contents
1. [**Chapter 0: The Target Model — Why Qwen2.5-0.5B Base?**](#ch0)
2. [**Chapter 1: Inspecting Base Model Autocomplete Chaos**](#ch1)
3. [**Chapter 2: Chat Templates & Tokenizer Special Tokens (ChatML)**](#ch2)
4. [**Chapter 3: Supervised Fine-Tuning (SFT) with Masked Loss**](#ch3)
5. [**Chapter 4: Parameter-Efficient Fine-Tuning (LoRA) From Scratch**](#ch4)
6. [**Chapter 5: Direct Preference Optimization (DPO) From Scratch**](#ch5)
7. [**Chapter 6: Reasoning RL with Verifiable Rewards (GRPO - DeepSeek Style)**](#ch6)
8. [**Chapter 7: Comprehensive Evaluation & Before/After Benchmarks**](#ch7)
9. [**Appendix: The Post-Training Career Roadmap & Industry Playbook**](#ch8)


<a id="ch0"></a>
# Chapter 0: The Target Model — Why Qwen2.5-0.5B Base?

### Model Selection Criteria:
1. **Base vs Instruct**:
   - Never learn post-training on an `-Instruct` or `-Chat` model! An instruct model has already undergone SFT, DPO, and RLHF. If you fine-tune it, you cannot see the dramatic phase transition from text predictor to assistant.
   - `Qwen/Qwen2.5-0.5B` is a true **Base Model**.
2. **Pedagogical & Compute Efficiency**:
   - **490 Million parameters**: Fits in ~1GB VRAM (bfloat16).
   - Trains in **minutes** on Apple Silicon (M1/M2/M3/M4 via `mps`) or a free Colab T4 / CPU.
   - Retains the exact same architectural components as the flagship 72B model: Rotary Positional Embeddings (RoPE), Grouped-Query Attention (GQA), SwiGLU activations, and RMSNorm.


In [ ]:
import os
import math
import copy
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Dict, List, Optional, Tuple, Any

# Set random seeds for reproducibility
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

# Hardware Detection: MPS (Apple Silicon), CUDA (Nvidia), or CPU
if torch.backends.mps.is_available():
    device = torch.device("mps")
    dtype = torch.float32  # MPS handles float32 most stably across all kernels
    print("🚀 Accelerated Device Detected: Apple Silicon (MPS)")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    print(f"🚀 Accelerated Device Detected: NVIDIA GPU ({torch.cuda.get_device_name(0)})")
else:
    device = torch.device("cpu")
    dtype = torch.float32
    print("⚠️ Accelerated Device NOT Detected: Running on CPU")


<a id="ch1"></a>
# Chapter 1: Inspecting Base Model Autocomplete Chaos

Before touching any weights, let's load `Qwen/Qwen2.5-0.5B` and test it with basic questions.

Notice what happens:
- A base model has no concept of a "user" or an "assistant".
- It treats your prompt as the opening sentences of an internet document, continuing with more text or questions.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B"

print(f"📦 Loading Tokenizer for {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Ensure pad token is configured
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"📦 Loading Model {MODEL_NAME}...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
    trust_remote_code=True
).to(device)
base_model.eval()

print("✅ Base Model Loaded Successfully!")
print(f"Total Parameters: {sum(p.numel() for p in base_model.parameters()):,}")


In [ ]:
def generate_text(
    model: nn.Module,
    tok: AutoTokenizer,
    prompt: str,
    max_new_tokens: int = 50,
    temperature: float = 0.7,
    top_p: float = 0.9,
    stop_strings: Optional[List[str]] = None
) -> str:
    """Generates text using standard autoregressive sampling."""
    inputs = tok(prompt, return_tensors="pt").to(device)
    input_ids = inputs["input_ids"]
    prompt_len = input_ids.shape[1]

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True if temperature > 0 else False,
            temperature=temperature if temperature > 0 else None,
            top_p=top_p if temperature > 0 else None,
            pad_token_id=tok.pad_token_id,
            eos_token_id=tok.eos_token_id
        )

    # Decode only the generated new tokens
    generated_ids = outputs[0, prompt_len:]
    return tok.decode(generated_ids, skip_special_tokens=False)

test_prompts = [
    "What is the capital of France?",
    "Write a Python function to check if a number is prime:",
    "Explain quantum computing in one simple sentence:"
]

print("=" * 70)
print("🔍 BASE MODEL RESPONSE (PRE-POSTTRAINING OBSERVED BEHAVIOR)")
print("=" * 70)

for p in test_prompts:
    res = generate_text(base_model, tokenizer, p, max_new_tokens=40, temperature=0.7)
    print(f"\n👉 PROMPT: {p}")
    print(f"🤖 RAW OUTPUT:\n{res}")
    print("-" * 50)


<a id="ch2"></a>
# Chapter 2: Chat Templates & Special Tokens (ChatML)

Why did the base model fail above? Because it does not know where the prompt ends and where an answer should begin!

### ChatML (Chat Markup Language)
Qwen models use the standard **ChatML** format, demarcated by special tokens:
```
<|im_start|>system
You are a helpful and harmless assistant.<|im_end|>
<|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant
The capital of France is Paris.<|im_end|>
```

### The Anatomy of Special Tokens
1. `<|im_start|>` (Token ID: `151644`): Signals the beginning of a role message.
2. `<|im_end|>` (Token ID: `151645`): Signals the end of a role message. When the assistant finishes talking, it **must** generate `<|im_end|>`.
3. If the model does not generate `<|im_end|>`, it will keep rambling forever!


In [ ]:
# Let's inspect Qwen's special tokens
special_tokens = ["<|im_start|>", "<|im_end|>", "<|endoftext|>"]
for st in special_tokens:
    token_id = tokenizer.convert_tokens_to_ids(st)
    print(f"Special token '{st}' -> Token ID: {token_id}")

# Let's see how apply_chat_template formats a multi-turn conversation
conversation = [
    {"role": "system", "content": "You are a concise AI tutor."},
    {"role": "user", "content": "What is 15 * 6?"},
    {"role": "assistant", "content": "15 * 6 is 90."}
]

formatted_chat = tokenizer.apply_chat_template(conversation, tokenize=False)
print("\nFormatted ChatML String:")
print(repr(formatted_chat))
print("\nRendered ChatML:")
print(formatted_chat)


<a id="ch3"></a>
# Chapter 3: Supervised Fine-Tuning (SFT) with Masked Loss

### The Golden Rule of SFT: Mask the Prompt!
If you calculate cross-entropy loss over the entire sequence, the model will spend gradients learning how to predict the **user's question** and **system prompt**.
- **We do not want the model to learn to predict the prompt!**
- We only want the model to learn to predict the **assistant's response**.

PyTorch's `nn.CrossEntropyLoss` has a built-in feature: `ignore_index = -100`. Any label set to `-100` contributes $0$ loss and $0$ gradients!

```
Token:     [<|im_start|>, user, \n, Hello, <|im_end|>, \n, <|im_start|>, assistant, \n, Hi, there, !, <|im_end|>]
Input IDs: [   151644,   872, 198, 9707,   151645, 198,   151644,       77091, 198, 6313, 1058, 0,   151645]
Labels:    [     -100,  -100, -100, -100,     -100, -100,     -100,       -100, -100, 6313, 1058, 0,   151645]
                      └───────────────────────────────┘                   └─────────────────────────┘
                           PROMPT TOKENS (MASKED)                           ASSISTANT TOKENS (TRAINED)
```


In [ ]:
class SFTDataset(torch.utils.data.Dataset):
    """Custom SFT dataset with precise prompt masking (labels = -100 on prompt)."""
    def __init__(self, data: List[Dict[str, str]], tokenizer: AutoTokenizer, max_length: int = 256):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.samples = []
        
        assistant_header = "<|im_start|>assistant\n"
        
        for item in data:
            instruction = item["instruction"]
            response = item["response"]
            
            # Format using ChatML
            prompt_chat = [
                {"role": "system", "content": "You are a helpful, precise assistant."},
                {"role": "user", "content": instruction}
            ]
            full_chat = prompt_chat + [
                {"role": "assistant", "content": response}
            ]
            
            # Tokenize prompt alone with generation header
            prompt_str = tokenizer.apply_chat_template(prompt_chat, tokenize=False, add_generation_prompt=True)
            full_str = tokenizer.apply_chat_template(full_chat, tokenize=False)
            
            prompt_ids = tokenizer.encode(prompt_str, add_special_tokens=False)
            full_ids = tokenizer.encode(full_str, add_special_tokens=False)
            
            # Pad or truncate
            if len(full_ids) > self.max_length:
                full_ids = full_ids[:self.max_length]
            
            input_ids = full_ids[:]
            labels = full_ids[:]
            
            # Mask all prompt tokens
            prompt_len = min(len(prompt_ids), len(labels))
            for i in range(prompt_len):
                labels[i] = -100  # PyTorch ignore_index
                
            self.samples.append({
                "input_ids": torch.tensor(input_ids, dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long)
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

def sft_collate_fn(batch: List[Dict[str, torch.Tensor]], pad_token_id: int) -> Dict[str, torch.Tensor]:
    """Pads sequences in batch to max sequence length in that batch."""
    max_len = max(len(s["input_ids"]) for s in batch)
    input_ids = []
    labels = []
    attention_mask = []
    
    for s in batch:
        seq_len = len(s["input_ids"])
        pad_len = max_len - seq_len
        
        inp = torch.cat([s["input_ids"], torch.full((pad_len,), pad_token_id, dtype=torch.long)])
        lbl = torch.cat([s["labels"], torch.full((pad_len,), -100, dtype=torch.long)])
        mask = torch.cat([torch.ones(seq_len, dtype=torch.long), torch.zeros(pad_len, dtype=torch.long)])
        
        input_ids.append(inp)
        labels.append(lbl)
        attention_mask.append(mask)
        
    return {
        "input_ids": torch.stack(input_ids),
        "labels": torch.stack(labels),
        "attention_mask": torch.stack(attention_mask)
    }

# Synthetic high-quality instruction dataset for demonstration
curated_sft_data = [
    {"instruction": "What is the capital of France?", "response": "The capital of France is Paris."},
    {"instruction": "What is 25 * 4?", "response": "25 * 4 = 100."},
    {"instruction": "Write a Python function to check if a number is prime:", "response": "def is_prime(n):\n    if n < 2:\n        return False\n    for i in range(2, int(n**0.5) + 1):\n        if n % i == 0:\n            return False\n    return True"},
    {"instruction": "Explain quantum computing in one simple sentence:", "response": "Quantum computing uses the strange properties of quantum physics—like superposition and entanglement—to process complex information exponentially faster than classical computers."},
    {"instruction": "Translate 'Hello, how are you?' to Spanish:", "response": "In Spanish, that translates to: '¡Hola! ¿Cómo estás?'"},
    {"instruction": "Who wrote Romeo and Juliet?", "response": "Romeo and Juliet was written by William Shakespeare."}
] * 8  # Replicated for training demonstration

sft_dataset = SFTDataset(curated_sft_data, tokenizer)
print(f"Created SFT Dataset with {len(sft_dataset)} samples.")
sample = sft_dataset[0]
print(f"Sample Input IDs shape: {sample['input_ids'].shape}")
print(f"Sample Labels shape:   {sample['labels'].shape}")
print(f"Number of masked prompt tokens (label = -100): {(sample['labels'] == -100).sum().item()}")
print(f"Number of trained target tokens:               {(sample['labels'] != -100).sum().item()}")
